# ATLANTIS for Relation Extraction — Qwen2 (Decoder-Only)

GPU hardware: A100 High-RAM required

1. `RUN_SIZE = "small"`  → Qwen2-0.5B / 1.5B — (~40 minute runtime)
2. `RUN_SIZE = "full"` → Qwen2-1.5B / 7B (~2 hour runtime)



## 1.1 Installation

In [ ]:
!pip install -q transformers datasets scikit-learn accelerate sentencepiece bitsandbytes
print("Done.")

## 1.2 Imports

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup
from datasets import load_dataset
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU:   {torch.cuda.get_device_name(0)}")
    print(f"VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1.3 Config

In [ ]:
# ── Run size — controls model pair and hyperparameters ───────────────────────
RUN_SIZE = "small"   # "small" or "full"

MODEL_CONFIGS = {
    "small":  {"small": "Qwen/Qwen2-0.5B", "large": "Qwen/Qwen2-1.5B"},
    "full": {"small": "Qwen/Qwen2-1.5B", "large": "Qwen/Qwen2-7B"},
}

INSTRUCT_CONFIGS = {
    "small":  "Qwen/Qwen2-0.5B-Instruct",
    "full":   "Qwen/Qwen2-1.5B-Instruct",
}

SMALL_MODEL = MODEL_CONFIGS[RUN_SIZE]["small"]
LARGE_MODEL = MODEL_CONFIGS[RUN_SIZE]["large"]
INSTRUCT_MODEL = INSTRUCT_CONFIGS[RUN_SIZE]

# Experiments can be run with off-the shelf instruct model or finetuned model
PR_MODE = "custom" # "custom" or "instruct"

# ── Dataset ───────────────────────────────────────────────────────────────────
DATASET = "semeval" # "semeval" or "conll2004"

SPLIT_PR = False # Option to sample subset of finetune data
SPLIT_RATIO = 0.5

# ── Training ──────────────────────────────────────────────────────────────────
MAX_INPUT_LEN  = 256
MAX_TARGET_LEN = 16
MAX_TOTAL_LEN  = MAX_INPUT_LEN + MAX_TARGET_LEN

# Hyperparameters scaled by run size
# Paper (<=13B): batch=2, grad_accum=8 → effective=16, lr=2e-5
TRAIN_CONFIGS = {
    "small":  {"batch": 8,  "grad_accum": 2,  "lr": 2e-5, "num_epochs": 2, "pr_epochs": 2},
    "full": {"batch": 1,  "grad_accum": 16,  "lr": 2e-5, "num_epochs": 2, "pr_epochs": 2},
}

BATCH_SIZE       = TRAIN_CONFIGS[RUN_SIZE]["batch"]
GRAD_ACCUM_STEPS = TRAIN_CONFIGS[RUN_SIZE]["grad_accum"]
NUM_EPOCHS       = TRAIN_CONFIGS[RUN_SIZE]["num_epochs"]
PR_EPOCHS        = TRAIN_CONFIGS[RUN_SIZE]["pr_epochs"]
LR               = TRAIN_CONFIGS[RUN_SIZE]["lr"]
WARMUP_RATIO     = 0.2

EVAL_LOGPROB_FALLBACK = False

# ── ATLANTIS weight computation ───────────────────────────────────────────────
LOGPROB_MODE = "sum"   # "mean" is modified for encoder-decoder models
                       # "sum"  same as paper
WEIGHT_CLIP  = 5.0

# ── Quick test ────────────────────────────────────────────────────────────────
QUICK_TEST = False

if QUICK_TEST:
    NUM_EPOCHS = 1
    PR_EPOCHS  = 2
    SUBSAMPLE  = 0.05
    print("QUICK_TEST mode: 1 epoch, 5% data")
else:
    SUBSAMPLE = 1.0
    print("Full experiment mode")

eff_batch = BATCH_SIZE * GRAD_ACCUM_STEPS
print(f"RUN_SIZE: {RUN_SIZE}")
print(f"  SMALL_MODEL: {SMALL_MODEL}")
print(f"  LARGE_MODEL: {LARGE_MODEL}")
print(f"  Batch: {BATCH_SIZE} x grad_accum {GRAD_ACCUM_STEPS} = effective {eff_batch}")
print(f"  LR: {LR}  |  NUM_EPOCHS: {NUM_EPOCHS}  |  PR_EPOCHS: {PR_EPOCHS}")
print(f"  LOGPROB_MODE: {LOGPROB_MODE!r}  |  WEIGHT_CLIP: {WEIGHT_CLIP}")

## 1.4 Dataset Loading

In [ ]:
SEMEVAL_CANONICAL = [
    "Cause-Effect",        # 0, 1
    "Component-Whole",     # 2, 3
    "Content-Container",   # 4, 5
    "Entity-Destination",  # 6, 7
    "Entity-Origin",       # 8, 9
    "Instrument-Agency",   # 10, 11
    "Member-Collection",   # 12, 13
    "Message-Topic",       # 14, 15
    "Product-Producer",    # 16, 17
    "Other",               # 18
]

CONLL_RELATIONS = ["Kill", "Live_In", "Located_In", "OrgBased_In", "Work_For"]

# CoNLL04 → SemEval approximate label mapping for transfer evaluation
CONLL_TO_SEMEVAL = {
    "Work_For":    "Instrument-Agency",
    "OrgBased_In": "Entity-Origin",
    "Live_In":     "Entity-Destination",
    "Located_In":  "Component-Whole",
    "Kill":        "Cause-Effect",
}

def semeval_label_to_str(label_id):
    if label_id == 18:
        return "Other"
    return SEMEVAL_CANONICAL[label_id // 2]

def load_examples(dataset_name, subsample=1.0):
    if dataset_name == "semeval":
        raw = load_dataset("sem_eval_2010_task_8")
        label_list = SEMEVAL_CANONICAL[:]
        label_options = ", ".join(label_list)
        def to_ex(item):
            return [{
                "input_text": (
                    f"Classify the semantic relation between the marked entities "
                    f"in this sentence: {item['sentence'].strip()}\n"
                    #f"Relation choices: {label_options}"
                ),
                "label_str": semeval_label_to_str(item["relation"]),
            }]
        train = [ex for item in raw["train"] for ex in to_ex(item)]
        test  = [ex for item in raw["test"]  for ex in to_ex(item)]

    elif dataset_name == "conll2004":
        raw = load_dataset("DFKI-SLT/conll04")
        label_list = CONLL_RELATIONS[:]
        label_options = ", ".join(CONLL_RELATIONS)

        def to_ex(item):
            tokens   = item["tokens"]
            entities = item["entities"]
            sentence = " ".join(tokens)
            examples = []
            for rel in item["relations"]:
                head_ent  = entities[rel["head"]]
                tail_ent  = entities[rel["tail"]]
                head_span = " ".join(tokens[head_ent["start"]:head_ent["end"]])
                tail_span = " ".join(tokens[tail_ent["start"]:tail_ent["end"]])
                examples.append({
                    "input_text": (
                        f"Classify the relation between <e1>{head_span}</e1> and <e2>{tail_span}</e2> "
                        f"in this sentence: {sentence}\n"
                        #f"Relation choices: {label_options}"
                    ),
                    "label_str": rel["type"],
                })
            return examples

        train = [ex for item in raw["train"] for ex in to_ex(item)]
        test  = [ex for item in raw["test"]  for ex in to_ex(item)]

    else:
        raise ValueError(f"Unknown dataset: {dataset_name!r}")

    if subsample < 1.0:
        n = max(100, int(len(train) * subsample))
        train = random.sample(train, n)

    return train, test, label_list

train_all, test_examples, LABEL_LIST = load_examples(DATASET, subsample=SUBSAMPLE)
LABEL2ID = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL  = {i: l for l, i in LABEL2ID.items()}

print(f"Dataset:  {DATASET}")
print(f"Train:    {len(train_all)}  Test: {len(test_examples)}")
print(f"Labels ({len(LABEL_LIST)}): {LABEL_LIST}")

if DATASET == "semeval":
    raw_check = load_dataset("sem_eval_2010_task_8")
    print("\nLabel mapping verification:")
    for i, name in enumerate(raw_check["train"].features["relation"].names):
        print(f"  {i:2d}: {name:35s} → {semeval_label_to_str(i)}")

elif DATASET == "conll2004":
    from collections import Counter
    dist = Counter(ex["label_str"] for ex in train_all)
    print(f"\nAvg relations per source sentence: {len(train_all)/922:.2f}")
    print("Label distribution:")
    for label, count in dist.most_common():
        print(f"  {label:<15} {count:>4}  ({count/len(train_all):.1%})")
    print(f"\nSample input: {train_all[0]['input_text']}")
    print(f"Sample label: {train_all[0]['label_str']}")

if SPLIT_PR:
    random.shuffle(train_all)
    split_idx   = int(len(train_all) * SPLIT_RATIO)
    train_pr    = train_all[:split_idx]   # p_r sees this
    train_large = train_all[split_idx:]   # p^L_b sees this
    print(f"Split mode: p_r trains on {len(train_pr)}, p^L_b trains on {len(train_large)}")
else:
    train_pr    = train_all
    train_large = train_all

## 1.5 Tokenizer


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(SMALL_MODEL, trust_remote_code=True)

# Qwen2 has no pad token by default — required for batched ops
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Tokenizer: {SMALL_MODEL}")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"EOS token: {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
print(f"PAD token: {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")

# Verify label tokenization — labels must tokenize to short sequences
print("\nLabel token counts:")
for label in LABEL_LIST:
    toks = tokenizer(label, add_special_tokens=False)["input_ids"]
    print(f"  {label:<25} {len(toks)} tokens: {toks}")

In [ ]:
# ── Check input lengths before training ──────────────────────────────────────
print("Checking input token lengths...")
lengths = [
    len(tokenizer(ex["input_text"], add_special_tokens=True)["input_ids"])
    for ex in train_all
]
pct_truncated = np.mean([l > MAX_INPUT_LEN for l in lengths])

print(f"  Max:    {max(lengths)} tokens")
print(f"  Mean:   {np.mean(lengths):.0f} tokens")
print(f"  Median: {int(np.median(lengths))} tokens")
print(f"  % > {MAX_INPUT_LEN} (truncated): {pct_truncated:.1%}")

if pct_truncated > 0.05:
    print(f"  Warning: >5% truncated — consider increasing MAX_INPUT_LEN to 512")
else:
    print(f"  OK: truncation acceptable")

## 1.6 Dataset Class & Data Loaders

For a causal LM we build a single concatenated sequence:
```
[prompt tokens] [label tokens] [EOS]
```
Labels tensor: `-100` for prompt positions (masked from loss), real token ids for label positions.

The loss is then mean NLL over label tokens only — equivalent to `log p(label | prompt)`.

In [ ]:
class REDataset(Dataset):
    """Causal LM RE dataset for Qwen2.

    Builds sequences: [prompt tokens][label tokens][EOS]
    Loss computed only over label tokens (prompt positions masked with -100).
    """
    def __init__(self, examples, label_source="gold", include_weights=False):
        self.examples        = examples
        self.label_source    = label_source
        self.include_weights = include_weights

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex    = self.examples[idx]
        label = ex["label_str"] if self.label_source == "gold" else ex["weak_label"]

        prompt_ids = tokenizer(
            ex["input_text"],
            max_length=MAX_INPUT_LEN,
            truncation=True,
            add_special_tokens=True,
        )["input_ids"]

        label_ids = tokenizer(
            label,
            max_length=MAX_TARGET_LEN,
            truncation=True,
            add_special_tokens=False,
        )["input_ids"]

        label_ids = label_ids + [tokenizer.eos_token_id]

        full_ids = prompt_ids + label_ids
        n_real   = len(full_ids)   # length before padding — all real tokens including EOS

        # Pad to MAX_TOTAL_LEN
        pad_len = MAX_TOTAL_LEN - n_real
        if pad_len > 0:
            full_ids = full_ids + [tokenizer.pad_token_id] * pad_len
        else:
            full_ids = full_ids[:MAX_TOTAL_LEN]
            n_real   = MAX_TOTAL_LEN   # truncated — all positions are real

        input_ids = torch.tensor(full_ids, dtype=torch.long)

        # Build mask from n_real, not token identity — correctly attends to EOS
        attention_mask = torch.zeros(MAX_TOTAL_LEN, dtype=torch.long)
        attention_mask[:n_real] = 1

        # Labels: -100 for prompt + padding, real ids for label tokens
        n_prompt    = len(prompt_ids)
        labels      = torch.full_like(input_ids, -100)
        label_start = n_prompt
        label_end   = min(n_prompt + len(label_ids), MAX_TOTAL_LEN)
        labels[label_start:label_end] = input_ids[label_start:label_end]

        item = {
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "labels":         labels,
            "n_prompt":       torch.tensor(n_prompt, dtype=torch.long),
        }
        if self.include_weights:
            item["weight"] = torch.tensor(ex["weight"], dtype=torch.float32)
        return item


def make_loaders(train_exs, label_source="gold", include_weights=False):
    train_ds = REDataset(train_exs, label_source=label_source,
                         include_weights=include_weights)
    test_ds  = REDataset(test_examples, label_source="gold")
    return (
        DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0),
        DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0),
    )

# Verify a few examples
ds = REDataset(train_all[:2], label_source="gold")
for item in ds:
    lbl = item["labels"].clone()
    lbl[lbl == -100] = tokenizer.pad_token_id
    decoded_label = tokenizer.decode(lbl, skip_special_tokens=True).strip()
    print(f"Label decoded from labels tensor: {decoded_label!r}")
    print(f"  Sequence length: {item['input_ids'].shape[0]}")
    print(f"  N prompt tokens: {item['n_prompt'].item()}")
    print(f"  Label token positions: {(item['labels'] != -100).sum().item()}")
    print()

print("Dataset class defined.")

## 1.7 Core Utilities: Log-prob and Evaluation

`compute_seq_logprob` for a causal LM:
1. Build `[prompt][label][EOS]` sequence
2. Mask prompt with -100
3. Forward pass → model returns mean NLL over unmasked (label) tokens
4. Return as mean or sum depending on `LOGPROB_MODE`

In [ ]:
_snap_stats = {"exact": 0, "substring": 0, "logprob": 0}


def compute_seq_logprob(model, input_text, label_str):
    """log p(label_str | input_text) for a causal LM.

    Builds [prompt][label][EOS], masks prompt with -100, reads loss over
    label tokens. Returns mean or sum per LOGPROB_MODE.
    """
    prompt_ids = tokenizer(
        input_text,
        max_length=MAX_INPUT_LEN,
        truncation=True,
        add_special_tokens=True,
        return_tensors="pt",
    )["input_ids"].to(DEVICE)

    label_ids = tokenizer(
        label_str,
        max_length=MAX_TARGET_LEN,
        truncation=True,
        add_special_tokens=False,
        return_tensors="pt",
    )["input_ids"].to(DEVICE)

    # Append EOS
    eos = torch.tensor([[tokenizer.eos_token_id]], device=DEVICE)
    label_ids_with_eos = torch.cat([label_ids, eos], dim=-1)

    full_ids = torch.cat([prompt_ids, label_ids_with_eos], dim=-1)
    n_prompt = prompt_ids.shape[1]
    n_label  = label_ids_with_eos.shape[1]

    # Labels: -100 for prompt, real ids for label+EOS
    labels = torch.full_like(full_ids, -100)
    labels[:, n_prompt:n_prompt + n_label] = label_ids_with_eos

    with torch.no_grad():
        out = model(input_ids=full_ids, labels=labels)

    # out.loss = mean NLL over non-(-100) tokens = mean NLL over label tokens
    mean_logprob = -out.loss.item()

    if LOGPROB_MODE == "sum":
        return mean_logprob * n_label
    else:
        return mean_logprob




def _snap_single(pred):
    """Exact then substring match. Returns label or None."""
    if pred in LABEL_LIST:
        return pred
    for label in LABEL_LIST:
        if label.lower() in pred.lower() or pred.lower() in label.lower():
            return label
    return None


def _logprob_snap_batch(model, input_texts):
    """Score all labels for each input, return best per input.
    One forward pass per label over all unresolved inputs.
    """
    results = [None] * len(input_texts)
    best_lp = [float("-inf")] * len(input_texts)

    for label in LABEL_LIST:
        for i, text in enumerate(input_texts):
            lp = compute_seq_logprob(model, text, label)
            if lp > best_lp[i]:
                best_lp[i] = lp
                results[i] = label
    return results


def print_snap_stats(reset=True):
    total = sum(_snap_stats.values()) or 1
    print("Label-snapping stats:")
    for tier, count in _snap_stats.items():
        print(f"  {tier:<12} {count:>5}  ({count/total:.1%})")
    if _snap_stats["logprob"] / total > 0.05:
        print("  ⚠ >5% fell through to log-prob — check generation quality")
    if reset:
        for k in _snap_stats:
            _snap_stats[k] = 0


def evaluate(model, loader):
    """Evaluate model, return (macro-F1, preds, golds).

    Generation: prompt-only input, slice off prompt from output to get label.
    Tier 1/2 (exact/substring): batched.
    Tier 3 (log-prob): per-label batched forward passes.
    """
    model.eval()
    preds, golds = [], []

    # For generation, use left-padding so the model sees the full prompt
    tokenizer.padding_side = "left"

    with torch.no_grad():
        for batch in tqdm(loader, desc="Eval", leave=False):
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            n_prompts      = batch["n_prompt"].tolist()

            # Decode gold labels from labels tensor
            lbl = batch["labels"].clone()
            lbl[lbl == -100] = tokenizer.pad_token_id
            batch_golds = [
                tokenizer.decode(l, skip_special_tokens=True).strip()
                for l in lbl
            ]

            # Generate — pass only prompt portion of each sequence
            # Reconstruct prompt-only inputs for generation
            prompt_ids_list = [
                ids[:n] for ids, n in zip(input_ids.cpu(), n_prompts)
            ]
            # Left-pad to same length for batched generate
            max_prompt_len = max(n_prompts)
            padded_prompts = torch.stack([
                torch.cat([
                    torch.full((max_prompt_len - len(p),), tokenizer.pad_token_id),
                    p
                ])
                for p in prompt_ids_list
            ]).to(DEVICE)
            prompt_mask = (padded_prompts != tokenizer.pad_token_id).long()

            gen_out = model.generate(
                input_ids=padded_prompts,
                attention_mask=prompt_mask,
                max_new_tokens=MAX_TARGET_LEN,
                num_beams=1,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

            # Slice off prompt tokens — only keep generated portion
            decoded = [
                tokenizer.decode(
                    gen_out[i][max_prompt_len:], skip_special_tokens=True
                ).strip()
                for i in range(len(n_prompts))
            ]

            # Tier 1 + 2
            batch_preds   = [None] * len(decoded)
            tier3_indices = []
            tier3_texts   = []

            decoded_inputs = [
                tokenizer.decode(ids[:n], skip_special_tokens=True)
                for ids, n in zip(input_ids.cpu(), n_prompts)
            ]

            for i, pred in enumerate(decoded):
                snapped = _snap_single(pred)
                if snapped is not None:
                    batch_preds[i] = snapped
                    _snap_stats["exact" if pred in LABEL_LIST else "substring"] += 1
                else:
                    tier3_indices.append(i)
                    tier3_texts.append(decoded_inputs[i])

            # Tier 3
            if tier3_indices:
                if EVAL_LOGPROB_FALLBACK:
                    _snap_stats["logprob"] += len(tier3_indices)
                    tier3_preds = _logprob_snap_batch(model, tier3_texts)
                    for i, pred in zip(tier3_indices, tier3_preds):
                        batch_preds[i] = pred
                else:
                    _snap_stats["logprob"] += len(tier3_indices)
                    for i in tier3_indices:
                        batch_preds[i] = LABEL_LIST[0]   # assign first label — counted as wrong if gold differs

            preds.extend(batch_preds)
            golds.extend(batch_golds)

    # Restore right-padding for training
    tokenizer.padding_side = "right"

    macro_f1 = f1_score(golds, preds, labels=LABEL_LIST, average="macro", zero_division=0)
    print_snap_stats(reset=True)
    return macro_f1, preds, golds


print("Utilities defined.")
print(f"LOGPROB_MODE={LOGPROB_MODE!r}")

## 1.8 ATLANTIS Loss

In [ ]:
def compute_per_example_ce(model, input_ids, attention_mask, labels):
    """Per-example mean CE in a single batched forward pass.
    """
    B = input_ids.size(0)
    per_ce = []

    out = model(input_ids=input_ids, attention_mask=attention_mask)

    # Apply causal shift manually
    shift_logits = out.logits[:, :-1, :].contiguous()  # (B, T-1, V)
    shift_labels = labels[:, 1:].contiguous()           # (B, T-1)

    B, T, V = shift_logits.shape
    ce = nn.CrossEntropyLoss(reduction="none", ignore_index=-100)
    per_tok = ce(
        shift_logits.view(B * T, V),
        shift_labels.view(B * T)
    ).view(B, T)  # (B, T-1)

    n_tok = (shift_labels != -100).float().sum(dim=-1).clamp(min=1)
    return per_tok.sum(dim=-1) / n_tok  # (B,)


def atlantis_loss(model, batch, mode="sft"):
    """Algorithm 1 line 6: L = mean_i(w_i * CE_i).
    Weights are pre-computed constants — never recomputed from model being trained.
    """
    input_ids      = batch["input_ids"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)
    labels         = batch["labels"].to(DEVICE)

    if mode == "sft":
        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        return out.loss, torch.ones(input_ids.size(0), device=DEVICE)

    weights   = batch["weight"].to(DEVICE)
    per_ce    = compute_per_example_ce(model, input_ids, attention_mask, labels)
    return (weights * per_ce).mean(), weights


print("ATLANTIS loss defined.")

## 1.9 Model Helpers

In [ ]:
def _load_model(model_name, frozen=False):
    """Load Qwen2 causal LM.
    """
    print(f"Loading {model_name}...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16 if DEVICE.type == "cuda" else torch.float32,
        trust_remote_code=True,
    ).to(DEVICE)

    if frozen:
        model.eval()
        for p in model.parameters():
            p.requires_grad_(False)

    print(f"  Parameters: {sum(p.numel() for p in model.parameters())/1e9:.2f}B")
    if DEVICE.type == "cuda":
        print(f"  VRAM after load: {torch.cuda.memory_allocated()/1e9:.2f} GB")
    return model


def load_frozen_small_model():
    print(f"Loading p^S_b: {SMALL_MODEL} (frozen)")
    return _load_model(SMALL_MODEL, frozen=True)


def load_frozen_large_model():
    print(f"Loading p^L_b: {LARGE_MODEL} (frozen, pre-computation only)")
    return _load_model(LARGE_MODEL, frozen=True)


def compute_and_normalize_weights(examples, key="weight"):
    """Pre-compute and self-normalize ATLANTIS weights using logsumexp."""
    from scipy.special import logsumexp

    log_w = np.array([
        ex["log_pl_b"] + ex["log_pr"] - ex["log_ps_b"]
        for ex in examples
    ])

    if LOGPROB_MODE == "sum":
        log_mean_w = logsumexp(log_w) - np.log(len(log_w))
        weights    = np.exp(log_w - log_mean_w)
    else:
        raw = np.exp(log_w)
        weights = raw / raw.mean()

    # Apply relaxed clip
    if WEIGHT_CLIP is not None:
        weights = np.clip(weights, a_min=0.1, a_max=WEIGHT_CLIP)

    # Global normalization to mean=1 (no per-class normalization)
    weights = weights / weights.mean()

    for ex, w in zip(examples, weights):
        ex[key] = float(w)

    return weights


def finetune_small_model(train_exs, tag="p_r", label_source="gold"):
    """Fine-tune Qwen2-small for PR_EPOCHS with gradient accumulation."""
    print(f"\nFine-tuning {tag} ({SMALL_MODEL}) for {PR_EPOCHS} epochs...")
    print(f"  Effective batch: {BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {BATCH_SIZE * GRAD_ACCUM_STEPS}")
    model = _load_model(SMALL_MODEL)
    train_loader, _ = make_loaders(train_exs, label_source=label_source)

    opt   = torch.optim.AdamW(model.parameters(), lr=LR)
    total = (len(train_loader) // GRAD_ACCUM_STEPS) * PR_EPOCHS
    sched = get_linear_schedule_with_warmup(opt, int(total * WARMUP_RATIO), total)

    for epoch in range(PR_EPOCHS):
        model.train(); losses = []
        opt.zero_grad()

        for step, batch in enumerate(tqdm(train_loader, desc=f"  {tag} epoch {epoch+1}/{PR_EPOCHS}")):
            out = model(
                input_ids=batch["input_ids"].to(DEVICE),
                attention_mask=batch["attention_mask"].to(DEVICE),
                labels=batch["labels"].to(DEVICE),
            )
            loss = out.loss / GRAD_ACCUM_STEPS
            loss.backward()
            losses.append(out.loss.item())

            if (step + 1) % GRAD_ACCUM_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step(); sched.step()
                opt.zero_grad()

        print(f"  Epoch {epoch+1}  loss={np.mean(losses):.4f}")

    model.eval()
    print(f"{tag} complete.")
    return model


def train_large_model(train_exs, label_source="gold", include_weights=False,
                      loss_mode="sft", tag=""):
    """Train fresh Qwen2-large (p_theta) with gradient accumulation.
    Effective batch = BATCH_SIZE x GRAD_ACCUM_STEPS, matching paper setup.
    """
    print(f"\n{'='*60}\n  {tag}\n{'='*60}")
    print(f"  Effective batch: {BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {BATCH_SIZE * GRAD_ACCUM_STEPS}")

    train_loader, test_loader = make_loaders(
        train_exs, label_source=label_source, include_weights=include_weights,
    )

    model = _load_model(LARGE_MODEL)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total = (len(train_loader) // GRAD_ACCUM_STEPS) * NUM_EPOCHS
    sched = get_linear_schedule_with_warmup(opt, int(total * WARMUP_RATIO), total)

    history = {"train_loss": [], "eval_f1": [], "weight_stats": []}

    for epoch in range(NUM_EPOCHS):
        model.train()
        losses, all_w = [], []
        opt.zero_grad()

        for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")):
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)

            if loss_mode == "atlantis":
                w      = batch["weight"].to(DEVICE)
                per_ce = compute_per_example_ce(model, input_ids, attention_mask, labels)
                loss   = (w * per_ce).mean()
            else: # Standard SFT
                out  = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = out.loss
                w    = torch.ones(input_ids.size(0), device=DEVICE)

            (loss / GRAD_ACCUM_STEPS).backward()
            losses.append(loss.item())
            all_w.extend(w.detach().cpu().tolist())

            if (step + 1) % GRAD_ACCUM_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                sched.step()
                opt.zero_grad()

        avg_loss = np.mean(losses)
        f1, _, _ = evaluate(model, test_loader)

        w_arr   = np.array(all_w)
        w_stats = {
            "mean": float(w_arr.mean()), "std": float(w_arr.std()),
            "min":  float(w_arr.min()),  "max": float(w_arr.max()),
            "pct_clip_high": float((w_arr >= WEIGHT_CLIP - 1e-6).mean()) if WEIGHT_CLIP else 0.0,
        }
        history["train_loss"].append(avg_loss)
        history["eval_f1"].append(f1)
        history["weight_stats"].append(w_stats)

        line = f"  Epoch {epoch+1}  loss={avg_loss:.4f}  macro-F1={f1:.4f}"
        if loss_mode == "atlantis":
            line += f"  w_mean={w_stats['mean']:.3f}  w_std={w_stats['std']:.3f}"
        print(line)

    history["final_f1"] = history["eval_f1"][-1]
    print(f"  -> Final F1: {history['final_f1']:.4f}")
    return history, model


def load_instruct_reference_model():
    """Load Qwen2-Instruct as frozen reference model p_r.

    Used when PR_MODE='instruct'.
    """
    print(f"Loading instruct reference model: {INSTRUCT_MODEL}")
    return _load_model(INSTRUCT_MODEL, frozen=True)


def get_reference_model(train_exs, tag="p_r", label_source="gold"):
    """Return reference model p_r — either finetuned or instruct depending on PR_MODE."""
    if PR_MODE == "instruct":
        return load_instruct_reference_model()
    else:
        return finetune_small_model(train_exs, tag=tag, label_source=label_source)


print("Model helpers defined.")
print(f"p_r: {PR_EPOCHS} epochs  |  p_theta: {NUM_EPOCHS} epochs")
print(f"Effective batch size: {BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {BATCH_SIZE * GRAD_ACCUM_STEPS}")

---
# Part 2 — Paradigm A: Paper Replication

All models use the same full clean SemEval training set.
ATLANTIS reweights examples but has no noise to correct.

| Condition | Data | Labels | Loss |
|-----------|------|--------|------|
| SFT | Full train | Clean | Standard CE |
| ATLANTIS | Full train | Clean | Weighted CE |

## 2.1 Fine-tune p_r on Full Training Set

In [ ]:
p_r_A = get_reference_model(train_pr, tag="p_r_A")

## 2.2 Evaluate p_r Quality

In [ ]:

_, test_loader_pr = make_loaders(test_examples, label_source="gold")
f1_pr, _, _ = evaluate(p_r_A, test_loader_pr)
print(f"p_r test F1: {f1_pr:.4f}")
print("If F1 < 0.3, increase PR_EPOCHS and retrain before computing weights.")

## 2.3 Load p^S_b

In [ ]:
p_s_b = load_frozen_small_model()

## 2.4 Pre-compute ATLANTIS Weights

All three log-prob terms computed from frozen models.
`p^L_b` is loaded here and unloaded after — the model trained in 2.6 is a fresh copy.

In [ ]:
print(f"Pre-computing weights for {len(train_large)} examples...")
print(f"LOGPROB_MODE={LOGPROB_MODE!r}  WEIGHT_CLIP={WEIGHT_CLIP}")
print()

p_l_b_frozen = load_frozen_large_model()
train_A = [dict(ex) for ex in train_large]

for ex in tqdm(train_A, desc="Computing log-probs"):
    ex["log_pr"]   = compute_seq_logprob(p_r_A,        ex["input_text"], ex["label_str"])
    ex["log_ps_b"] = compute_seq_logprob(p_s_b,        ex["input_text"], ex["label_str"])
    ex["log_pl_b"] = compute_seq_logprob(p_l_b_frozen, ex["input_text"], ex["label_str"])

weights = compute_and_normalize_weights(train_A)

print(f"\nSample log_pr:   {train_A[0]['log_pr']:.3f}")
print(f"Sample log_ps_b: {train_A[0]['log_ps_b']:.3f}")
print(f"Sample log_pl_b: {train_A[0]['log_pl_b']:.3f}")
print()
print(f"Normalized weight mean: {weights.mean():.4f}")
print(f"Normalized weight std:  {weights.std():.4f}")
print(f"Normalized weight min:  {weights.min():.4f}")
print(f"Normalized weight max:  {weights.max():.4f}")
print()

# Class bias check
by_label = {}
for ex in train_A:
    by_label.setdefault(ex["label_str"], []).append(ex["weight"])
print("Mean weight by relation type:")
for l, ws in sorted(by_label.items(), key=lambda x: -np.mean(x[1])):
    print(f"  {l:<25} mean={np.mean(ws):.3f}  std={np.std(ws):.3f}  n={len(ws)}")
print()
print("Healthy: roughly uniform across classes. If skewed, try LOGPROB_MODE='mean'.")

## 2.5 Free Frozen Models

In [ ]:
del p_r_A, p_s_b, p_l_b_frozen
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 2.6 Run Paradigm A Experiments

In [ ]:
hist_A_sft, model_A_sft = train_large_model(
    train_A,
    label_source="gold",
    include_weights=False,
    loss_mode="sft",
    tag="Paradigm A — SFT baseline",
)

In [ ]:
del model_A_sft
if torch.cuda.is_available(): torch.cuda.empty_cache()

hist_A_atlantis, model_A_atlantis = train_large_model(
    train_A,
    label_source="gold",
    include_weights=True,
    loss_mode="atlantis",
    tag="Paradigm A — ATLANTIS",
)

## 2.7 Prediction Distribution Check

In [ ]:
from collections import Counter
_, test_loader = make_loaders(train_A, label_source="gold")
_, preds_sft,  golds = evaluate(model_A_sft,      test_loader) if "model_A_sft"      in dir() else (None, None, None)
_, preds_atl, golds  = evaluate(model_A_atlantis,  test_loader)

print("ATLANTIS predicted distribution:")
for label, count in Counter(preds_atl).most_common():
    print(f"  {label:<25} {count}")

## 2.8 Paradigm A Results

In [ ]:
results_A = {"SFT": hist_A_sft, "ATLANTIS": hist_A_atlantis}
COLORS_A  = {"SFT": "#e74c3c", "ATLANTIS": "#3498db"}
epochs_x  = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f"Paradigm A — {LARGE_MODEL} (Decoder-Only)",
             fontsize=13, fontweight="bold")

ax = axes[0]
for name, h in results_A.items():
    ax.plot(epochs_x, h["eval_f1"], marker="o", label=name, color=COLORS_A[name])
ax.set_title("Macro-F1 by Epoch"); ax.set_xlabel("Epoch"); ax.set_ylabel("Macro-F1")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
for name, h in results_A.items():
    ax.plot(epochs_x, h["train_loss"], marker="s", label=name, color=COLORS_A[name])
ax.set_title("Training Loss"); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[2]
names = list(results_A.keys())
f1s   = [results_A[n]["final_f1"] for n in names]
bars  = ax.bar(names, f1s, color=[COLORS_A[n] for n in names],
               edgecolor="black", linewidth=0.7, width=0.4)
ax.set_ylim(max(0, min(f1s) - 0.05), min(1.0, max(f1s) + 0.05))
for bar, val in zip(bars, f1s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f"{val:.3f}", ha="center", va="bottom", fontsize=11)
ax.set_title("Final Macro-F1"); ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("paradigm_A_qwen_results.png", dpi=150, bbox_inches="tight")
plt.show()

delta = hist_A_atlantis["final_f1"] - hist_A_sft["final_f1"]
print(f"SFT:       {hist_A_sft['final_f1']:.4f}")
print(f"ATLANTIS:  {hist_A_atlantis['final_f1']:.4f}")
print(f"Delta:     {delta:+.4f}")

---
# Part 3 — Paradigm B: LLM-Generated Weak Labels

In [ ]:
# ── Upload weak labels file from local machine ───────────────────────────────
# Supports CSV or JSON. Expected columns/keys:
#   - "sentence" or "input_text" : the raw sentence (used to align with train_all)
#   - "label" or "relation"      : integer 0-18 (SemEval label ID)
#
# If your file has different column names, update WEAK_TEXT_COL / WEAK_LABEL_COL below.

from google.colab import files
import pandas as pd
import json, os

WEAK_TEXT_COL  = "sentence"    # column containing the sentence / input text
WEAK_LABEL_COL = "label"       # column containing the integer label (0-18)

print("Upload your weak labels file (CSV or JSON)...")
uploaded = files.upload()
weak_file = list(uploaded.keys())[0]
print(f"Uploaded: {weak_file}")

ext = os.path.splitext(weak_file)[1].lower()
if ext == ".csv":
    weak_df = pd.read_csv(weak_file)
elif ext in (".json", ".jsonl"):
    try:
        weak_df = pd.read_json(weak_file)
    except ValueError:
        weak_df = pd.read_json(weak_file, lines=True)
else:
    raise ValueError(f"Unsupported file type: {ext}. Use CSV or JSON.")

print(f"Loaded {len(weak_df)} rows. Columns: {list(weak_df.columns)}")
print(weak_df.head(3))

assert WEAK_TEXT_COL  in weak_df.columns, f"Column '{WEAK_TEXT_COL}' not found. Set WEAK_TEXT_COL."
assert WEAK_LABEL_COL in weak_df.columns, f"Column '{WEAK_LABEL_COL}' not found. Set WEAK_LABEL_COL."

weak_df["weak_label_str"] = weak_df[WEAK_LABEL_COL].astype(int).map(semeval_label_to_str)

# Verify no unmapped labels
unmapped = weak_df["weak_label_str"].isna().sum()
assert unmapped == 0, f"{unmapped} labels could not be mapped — check values are 0-18"

if "gold_label" in weak_df.columns:
    weak_df["gold_label_str"] = weak_df["gold_label"].astype(int).map(semeval_label_to_str)
    acc = (weak_df["weak_label_str"] == weak_df["gold_label_str"]).mean()
    print(f"\nWeak label accuracy vs gold: {acc:.1%}")
    print(f"Noise rate: {1-acc:.1%}")
else:
    print("\nNo gold_label column found — accuracy unknown.")

print(f"\nWeak label distribution:")
print(weak_df["weak_label_str"].value_counts())

In [ ]:
# Matches on input_text. Weak label file may contain raw sentences — we
# reconstruct input_text using the same prompt template as load_examples().

def normalize_for_lookup(text):
    """Strip whitespace for robust matching."""
    return text.strip()

# Build lookup from sentence → weak label string
# Try matching on input_text first, fall back to raw sentence
weak_lookup = {}
for _, row in weak_df.iterrows():
    raw_text  = normalize_for_lookup(str(row[WEAK_TEXT_COL]))
    weak_label = row["weak_label_str"]

    # Build the same prompt template used in load_examples
    label_options = ", ".join(SEMEVAL_CANONICAL)
    constructed_input = (
        "Classify the semantic relation between the marked entities "
        f"in this sentence: {raw_text}\n"
        #f"Relation choices: {label_options}"
    )
    weak_lookup[normalize_for_lookup(constructed_input)] = weak_label
    # Also index by raw sentence as fallback
    weak_lookup[raw_text] = weak_label

# Align with train_large
weak_B = []
unmatched = 0
for ex in train_large:
    key = normalize_for_lookup(ex["input_text"])
    weak_label = weak_lookup.get(key)

    if weak_label is None:
        # Fallback: try matching just the sentence portion (after the prompt prefix)
        for raw_key, lbl in weak_lookup.items():
            if raw_key in key:
                weak_label = lbl
                break

    if weak_label is None:
        unmatched += 1
        continue

    weak_B.append({
        "input_text": ex["input_text"],
        "label_str":  ex["label_str"],    # gold label — used for upper-bound eval
        "weak_label": weak_label,          # GPT label — used for training
    })

print(f"Matched:   {len(weak_B)} / {len(train_large)} examples")
print(f"Unmatched: {unmatched}")

if unmatched > len(train_large) * 0.1:
    print("Warning: >10% unmatched — check that WEAK_TEXT_COL contains the raw sentences")
    print(f"  Sample train input_text: {train_large[0]['input_text'][:100]}")
    print(f"  Sample weak file text:   {weak_df[WEAK_TEXT_COL].iloc[0][:100]}")

if weak_B:
    correct = sum(ex["weak_label"] == ex["label_str"] for ex in weak_B)
    print(f"\nNoise rate (matched examples): {1 - correct/len(weak_B):.1%}")
    print(f"Weak label accuracy:           {correct/len(weak_B):.1%}")

    from collections import Counter
    print("\nWeak label distribution:")
    dist = Counter(ex["weak_label"] for ex in weak_B)
    for label, count in dist.most_common():
        print(f"  {label:<25} {count:>4}  ({count/len(weak_B):.1%})")

In [ ]:
TARGET_NOISE_RATE = 0.20

def mix_labels(weak_examples, target_noise_rate):
    """Replace weak labels with gold labels to hit target noise rate.

    Current noise rate is estimated from examples that have both
    label_str (gold) and weak_label (GPT).
    """
    current_noise = sum(ex["weak_label"] != ex["label_str"] for ex in weak_examples) / len(weak_examples)
    print(f"Current noise rate:  {current_noise:.1%}")
    print(f"Target noise rate:   {target_noise_rate:.1%}")

    if target_noise_rate >= current_noise:
        print("Target >= current — no mixing needed.")
        return [dict(ex) for ex in weak_examples]

    # How many examples need to be corrected?
    n_total      = len(weak_examples)
    n_noisy      = int(current_noise * n_total)      # currently wrong
    n_target_noisy = int(target_noise_rate * n_total) # want this many wrong
    n_to_fix     = n_noisy - n_target_noisy

    print(f"Correcting {n_to_fix} examples (replacing weak label with gold)...")

    # Find currently wrong examples and randomly fix n_to_fix of them
    wrong_idx = [i for i, ex in enumerate(weak_examples) if ex["weak_label"] != ex["label_str"]]
    fix_idx   = set(random.sample(wrong_idx, n_to_fix))

    mixed = []
    for i, ex in enumerate(weak_examples):
        ex_copy = dict(ex)
        if i in fix_idx:
            ex_copy["weak_label"] = ex_copy["label_str"]   # replace with gold
        mixed.append(ex_copy)

    # Verify
    actual_noise = sum(ex["weak_label"] != ex["label_str"] for ex in mixed) / len(mixed)
    print(f"Actual noise after mixing: {actual_noise:.1%}")
    return mixed

# Generate mixed datasets at different noise rates for ablation
noise_rates  = [0.10, 0.20, 0.30, 0.40, 0.50]
mixed_by_noise = {r: mix_labels(weak_B, r) for r in noise_rates}

# Use one for Paradigm B experiments
weak_B_mixed = mixed_by_noise[TARGET_NOISE_RATE]
print(f"\nUsing weak_B_mixed with {TARGET_NOISE_RATE:.0%} noise for Paradigm B.")

In [ ]:
p_r_B = finetune_small_model(weak_B_mixed, tag="p_r_B (GPT-labeled)", label_source="weak")

_, test_loader_pr_B = make_loaders(test_examples, label_source="gold")
f1_pr_B, _, _ = evaluate(p_r_B, test_loader_pr_B)
print(f"p_r_B test F1: {f1_pr_B:.4f}")

In [ ]:
p_s_b_B        = load_frozen_small_model()
p_l_b_frozen_B = load_frozen_large_model()

for ex in tqdm(weak_B_mixed, desc="Computing log-probs"):
    ex["log_pr"]   = compute_seq_logprob(p_r_B,          ex["input_text"], ex["weak_label"])
    ex["log_ps_b"] = compute_seq_logprob(p_s_b_B,        ex["input_text"], ex["weak_label"])
    ex["log_pl_b"] = compute_seq_logprob(p_l_b_frozen_B, ex["input_text"], ex["weak_label"])
    ex["influence_ratio"] = ex["log_pr"] - ex["log_ps_b"]

weights = compute_and_normalize_weights(weak_B_mixed)

print(f"Normalized weight mean: {weights.mean():.4f}")
print(f"Normalized weight std:  {weights.std():.4f}")
print(f"Normalized weight min:  {weights.min():.4f}")
print(f"Normalized weight max:  {weights.max():.4f}")

by_label = {}
for ex in weak_B_mixed:
    by_label.setdefault(ex["label_str"], []).append(ex["weight"])
print("\nMean weight by relation type:")
for l, ws in sorted(by_label.items(), key=lambda x: -np.mean(x[1])):
    print(f"  {l:<25} mean={np.mean(ws):.3f}  std={np.std(ws):.3f}  n={len(ws)}")

In [ ]:
del p_r_B, p_s_b_B
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
hist_B_uniform, model_B_uniform = train_large_model(
    weak_B_mixed, label_source="weak", include_weights=False, loss_mode="sft",
    tag="Paradigm B — Uniform (GPT labels, w=1)",
)

In [ ]:
del model_B_uniform; torch.cuda.empty_cache()

hist_B_atlantis, model_B_atlantis = train_large_model(
    weak_B_mixed, label_source="weak", include_weights=True, loss_mode="atlantis",
    tag="Paradigm B — ATLANTIS (GPT labels, weighted CE)",
)

In [ ]:
results_B = {"uniform": hist_B_uniform, "atlantis": hist_B_atlantis}
COLORS_B  = {"uniform": "#e74c3c", "atlantis": "#3498db"}
epochs_x  = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f"Paradigm B — {LARGE_MODEL} + GPT Weak Labels",
             fontsize=13, fontweight="bold")

ax = axes[0]
for name, h in results_B.items():
    ax.plot(epochs_x, h["eval_f1"], marker="o", label=name, color=COLORS_B[name])
ax.set_title("Macro-F1 by Epoch"); ax.set_xlabel("Epoch"); ax.set_ylabel("Macro-F1")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1]
for name, h in results_B.items():
    ax.plot(epochs_x, h["train_loss"], marker="s", label=name, color=COLORS_B[name])
ax.set_title("Training Loss"); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[2]
names_b = list(results_B.keys())
f1s_b   = [results_B[n]["final_f1"] for n in names_b]
bars    = ax.bar(names_b, f1s_b, color=[COLORS_B[n] for n in names_b],
                 edgecolor="black", linewidth=0.7)
ax.set_ylim(max(0, min(f1s_b) - 0.05), min(1.0, max(f1s_b) + 0.05))
for bar, val in zip(bars, f1s_b):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f"{val:.3f}", ha="center", va="bottom", fontsize=9)
ax.set_title("Final Macro-F1"); ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("paradigm_B_qwen_results.png", dpi=150, bbox_inches="tight")
plt.show()

f1_uniform  = hist_B_uniform["final_f1"]
f1_atlantis = hist_B_atlantis["final_f1"]

print(f"\n{'Condition':<12} {'F1':>8}")
print("-" * 22)
for name in ["uniform", "atlantis"]:
    print(f"{name:<12} {results_B[name]['final_f1']:>8.4f}")
print()